# AlphaZero Chess: Supervised Pre-training on Colab (A100)

Run all cells top-to-bottom. Expects:
- A Google Drive folder containing `lichess_elite_2024-06.pgn` and `stockfish_scores copy.npy`.
- A GitHub repo for the project, OR you upload the source files manually.

Outputs (checkpoints, logs) are written back to Drive so they persist across runtime restarts.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure paths

Edit `DRIVE_PROJECT_DIR` to point to a folder in your Drive containing the data files.

In [ ]:
import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/AlphaZero'   # <-- edit me
GAMES_PATH = f'{DRIVE_PROJECT_DIR}/lichess_elite_2024-06.pgn'
EVALS_PATH = f'{DRIVE_PROJECT_DIR}/stockfish_scores copy.npy'
CHECKPOINT_DIR = f'{DRIVE_PROJECT_DIR}/checkpoints'
LOG_DIR = f'{DRIVE_PROJECT_DIR}/logs'

for path in (GAMES_PATH, EVALS_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing data file: {path}')
print('Data found.')

## 4. Get the project source

**Option A: clone from GitHub** — replace the URL with your repo.

In [ ]:
%cd /content
![ -d AlphaZero ] && rm -rf AlphaZero
!git clone https://github.com/NeoAcar/AlphaZero.git
%cd /content/AlphaZero

**Option B (alternative):** if you don't want to push to GitHub, upload the .py files via the file browser into `/content/AlphaZero/` and skip the clone cell.

## 5. Install dependencies

Colab already has torch+CUDA pre-installed; we only need a few extras.

In [ ]:
!pip install -q 'python-chess>=1.11' 'tensorboard>=2.18' 'tqdm>=4.66'

## 6. (Optional) Start TensorBoard

You can monitor loss curves while training runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR

## 7. Train

Tune `MAX_GAMES` and `EPOCHS` for your time budget. On A100:
- 10K games × 10 epochs ≈ 15 minutes
- 50K games × 10 epochs ≈ 60–90 minutes

In [ ]:
MAX_GAMES = 10000
EPOCHS = 10
BATCH_SIZE = 512   # bump from 256 since A100 has plenty of memory
LEARNING_RATE = 1e-4   # higher than your original 1e-5; A100 trains fast
LABEL_SMOOTHING = 0.1

import shlex
cmd = (
    f'python train.py '
    f'--games-path {shlex.quote(GAMES_PATH)} '
    f'--evals-path {shlex.quote(EVALS_PATH)} '
    f'--checkpoint-dir {shlex.quote(CHECKPOINT_DIR)} '
    f'--log-dir {shlex.quote(LOG_DIR)} '
    f'--max-games {MAX_GAMES} '
    f'--epochs {EPOCHS} '
    f'--batch-size {BATCH_SIZE} '
    f'--learning-rate {LEARNING_RATE} '
    f'--label-smoothing {LABEL_SMOOTHING}'
)
print(cmd)
!{cmd}

## 8. Verify checkpoint saved to Drive

In [ ]:
!ls -la $CHECKPOINT_DIR

## 9. (Optional) Smoke-test the trained model with self-play

Verifies the checkpoint loads and plays a short game.

In [ ]:
import glob
ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/model_epoch_*.pth'))
if not ckpts:
    raise RuntimeError('No checkpoint found.')
ckpt = ckpts[-1]
print(f'Using {ckpt}')
!python play.py bot --checkpoint {ckpt} --sims 50 --truncation 20 --delay 0